# Stage 3 — Patient-grouped splits and train-only preprocessing

This notebook freezes the MIMIC-IV development/internal-test split and the five development CV folds. It also verifies that static SMOTE and hourly imputation/scaling are fitted on a training fold only.

> **PhysioNet DUA:** run this notebook only in the controlled Kaggle/Colab environment containing the protected Stage-2 artifacts. Never publish the patient-level split assignment file or notebook outputs containing IDs.

## Protocol

1. Reserve 20% of patients as the untouched internal test set.
2. Assign the remaining patients to five `StratifiedGroupKFold` validation folds.
3. Fit imputation, scaling, one-hot encoding, and SMOTE inside each static-model training fold.
4. Fit hourly medians/scaling and the LSTM class weight on each sequence-model training fold.
5. Use development OOF predictions later to choose the F1-optimal threshold; never tune it on the test set.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import N_CV_FOLDS, RANDOM_SEED, TEST_SIZE  # noqa: E402
from src.features import HOURLY_TENSOR_PATH, STATIC_FEATURES_PATH  # noqa: E402
from src.splits import (  # noqa: E402
    PatientSplits,
    SPLIT_ASSIGNMENTS_PATH,
    SPLIT_SUMMARY_PATH,
    build_split_summary,
    build_static_preprocessor,
    compute_pos_weight,
    fit_hourly_preprocessor,
    load_hourly_features,
    run_stage3,
    smote_resample,
    validate_artifact_alignment,
    validate_patient_splits,
)

print(f'Project root: {ROOT}')
print(f'Protocol: test_size={TEST_SIZE:.0%}, folds={N_CV_FOLDS}, seed={RANDOM_SEED}')

## 1. Verify protected Stage-2 inputs

Only file existence is shown. Do not display patient rows or identifiers.

In [ ]:
required_paths = [STATIC_FEATURES_PATH, HOURLY_TENSOR_PATH]
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Stage 2 artifacts are missing. Run notebooks/02_features.ipynb first: '
        + ', '.join(str(path) for path in missing)
    )
print('Protected Stage-2 inputs found.')

## 2. Freeze the holdout and CV assignments

The assignment Parquet is a patient-level derivative and remains under the gitignored `data/processed/` directory. The CSV contains aggregate counts only. Re-running with the locked seed produces the same assignment.

In [ ]:
artifacts = run_stage3()
print(f'Rows: {artifacts.n_rows:,}')
print(f'Unique patients: {artifacts.n_subjects:,}')
print(f'Development rows: {artifacts.n_dev:,}')
print(f'Internal-test rows: {artifacts.n_test:,}')
print(f'Development folds: {artifacts.n_folds}')
print(f'Protected assignments: {artifacts.assignments_path}')
print(f'Aggregate summary: {artifacts.summary_path}')

In [ ]:
static = pd.read_parquet(STATIC_FEATURES_PATH)
hourly = load_hourly_features(HOURLY_TENSOR_PATH)
splits = PatientSplits(pd.read_parquet(SPLIT_ASSIGNMENTS_PATH))
validate_artifact_alignment(static, hourly)
validate_patient_splits(static, splits)
summary = build_split_summary(splits)
display(summary.style.format({'prevalence': '{:.3%}'}))

## 3. Verify patient isolation

These assertions inspect protected identifiers in memory but print only pass/fail messages.

In [ ]:
assignments = splits.assignments
dev_subjects = set(assignments.loc[assignments['split'].eq('dev'), 'subject_id'])
test_subjects = set(assignments.loc[assignments['split'].eq('test'), 'subject_id'])
assert dev_subjects.isdisjoint(test_subjects)
for fold_number, (train_idx, validation_idx) in enumerate(splits.iter_cv()):
    train_subjects = set(static.iloc[train_idx]['subject_id'])
    validation_subjects = set(static.iloc[validation_idx]['subject_id'])
    assert train_subjects.isdisjoint(validation_subjects), fold_number
print('Patient-level holdout and all CV leakage checks passed.')

## 4. Audit one static-model fold

This cell demonstrates the exact order used in Stage 4: fit the preprocessor on the training fold, transform validation without refitting, and apply SMOTE only to the transformed training fold. Race remains available for the later audit but is excluded from model inputs.

In [ ]:
train_idx, validation_idx = next(splits.iter_cv())
static_train = static.iloc[train_idx]
static_validation = static.iloc[validation_idx]

static_preprocessor = build_static_preprocessor(static_train)
x_train = static_preprocessor.fit_transform(static_train)
x_validation = static_preprocessor.transform(static_validation)
y_train = static_train['label'].to_numpy(dtype=np.int8)
x_train_smote, y_train_smote = smote_resample(x_train, y_train)

before = np.bincount(y_train, minlength=2)
after = np.bincount(y_train_smote, minlength=2)
print(f'Train/validation transformed shapes: {x_train.shape} / {x_validation.shape}')
print(f'Training class counts before SMOTE: negative={before[0]:,}, positive={before[1]:,}')
print(f'Training class counts after SMOTE:  negative={after[0]:,}, positive={after[1]:,}')
print('Validation rows were transformed only and were not resampled.')
del x_train, x_validation, x_train_smote, y_train_smote

## 5. Audit one LSTM fold

Hourly medians and scaling parameters are learned from the training fold only. The untouched observation mask is appended after scaling, giving an LSTM input of `[patients, 6, 58]` for 29 value channels plus 29 mask channels. Sequence data are not passed through SMOTE.

In [ ]:
hourly_preprocessor = fit_hourly_preprocessor(hourly.values[train_idx])
lstm_train = hourly_preprocessor.transform(
    hourly.values[train_idx], hourly.mask[train_idx]
)
lstm_validation = hourly_preprocessor.transform(
    hourly.values[validation_idx], hourly.mask[validation_idx]
)
pos_weight = compute_pos_weight(hourly.labels[train_idx])
print(f'LSTM train shape: {lstm_train.shape}')
print(f'LSTM validation shape: {lstm_validation.shape}')
print(f'Training-fold BCE positive-class weight: {pos_weight:.4f}')
del lstm_train, lstm_validation

## 6. Stage 3 acceptance checklist

- Development and internal-test patient sets are disjoint.
- Every development patient belongs to exactly one validation fold.
- All folds contain both outcome classes and have similar prevalence.
- Static imputation, scaling, one-hot encoding, and SMOTE are train-fold only.
- Hourly medians/scaling and `pos_weight` are train-fold only; no sequence SMOTE.
- The 20% internal test set remains untouched until Stage 5.
- Stage 4 must collect one OOF probability per development row for later threshold selection.

In [ ]:
assert SPLIT_ASSIGNMENTS_PATH.is_file()
assert SPLIT_SUMMARY_PATH.is_file()
assert len(splits.dev_indices) + len(splits.test_indices) == len(static)
assert set(splits.assignments.loc[splits.assignments['split'].eq('dev'), 'cv_fold']) == set(range(N_CV_FOLDS))
print('Stage 3 acceptance checks passed. The frozen assignments are ready for Stage 4.')